In [1]:
import pandas as pd
import pathlib
import math

In [2]:
appname = "trips"
home_dir = pathlib.Path.home()
bronze = home_dir / "data" / "bronze" / appname
silver = home_dir / "data" / "silver" / appname
gold = home_dir / "data" / "gold" / appname

bronze.mkdir(parents=True, exist_ok=True)
silver.mkdir(parents=True, exist_ok=True)
gold.mkdir(parents=True, exist_ok=True)


In [3]:
poi = pd.read_csv("/Users/ambikha/PycharmProjects/trips/data/poi.csv")
poi.head()

,name,latitude_radian,longitude_radian,num_links,links,num_categories,categories
0,"YAYCHI, WEST AZERBAIJAN",0.683175,0.778053,13,Baba Jik Rural District; West Azerbaijan Provi...,1,POPULATED PLACES IN CHALDORAN COUNTY
1,MOUNT FISKE GLACIER,0.648196,-2.071114,9,Mount Fiske; Mount Warlow Glacier; U.S. state;...,3,GLACIERS OF THE SIERRA NEVADA (U.S.); GLACIERS...
2,ALATONA,0.258356,-0.103606,10,Diabaly; Alatona Irrigation Project; Mali; Nio...,2,POPULATED PLACES IN SÉGOU REGION; IRRIGATION P...
3,PEMBA ISLAND,-0.090175,0.694350,43,Malaysia Airlines Flight 370; Arusha; Chake Ch...,5,PEMBA ISLAND; ISLANDS OF TANZANIA; ISLANDS OF ...
4,MBOLO,0.149517,0.359829,6,UTC; Sub-prefectures of the Central African Re...,2,N'DÉLÉ; POPULATED PLACES IN BAMINGUI-BANGORAN


In [4]:
# Approximate latitude and longitude bounds for Nevada in radians including Area 51 in southern Nevada
# Nevada: lat 37.2000°N to 37.3000°N, lon 115.9000°W to 115.7000°W

lat_min = math.radians(33)
lat_max = math.radians(37.5)
lon_min = math.radians(-116)
lon_max = math.radians(-115.8)
"""lat_min = math.radians(37.2000)
lat_max = math.radians(37.3000)
lon_min = math.radians(-115.9000)
lon_max = math.radians(-115.7000)"""


poi["latitude_degree"] = poi["latitude_radian"].apply(math.degrees)
poi["longitude_degree"] = poi["longitude_radian"].apply(math.degrees)

poi_nv = poi[
    (poi["latitude_radian"] >= lat_min)
    & (poi["latitude_radian"] <= lat_max)
    & (poi["longitude_radian"] >= lon_min)
    & (poi["longitude_radian"] <= lon_max)
].head(60)


In [5]:
print(f"POIs in Nv: {len(poi_nv)} locations\n")

from IPython.display import display

poi_display = poi_nv[["name", "latitude_degree", "longitude_degree"]].copy()

poi_display["latitude_degree"] = poi_display["latitude_degree"].round(5)
poi_display["longitude_degree"] = poi_display["longitude_degree"].round(5)

poi_display = poi_display.rename(
    columns={
        "name": "POI Name",
        "latitude_degree": "Latitude (deg)",
        "longitude_degree": "Longitude (deg)",
    }
)

display(
    poi_display.reset_index(drop=True).style.set_properties(
        **{
            "background-color": "#f5f5f5",
            "border": "1px solid black",
            "text-align": "left",
        }
    )
)

print("\n`First 6 POIs:")
for idx, row in poi_nv.iloc[:6].iterrows():
    print(f"- {row['name']}: ({row['latitude_radian']:.5f}, {row['longitude_radian']:.5f})")


POIs in Nv: 24 locations



,POI Name,Latitude (deg),Longitude (deg)
0,MOJAVE TRAILS NATIONAL MONUMENT,34.600000,-116.000000
1,"NORTH SHORE, CALIFORNIA",33.512780,-115.927220
2,"SQUEAKY SPRINGS, CALIFORNIA",33.257780,-115.948330
3,BURIED MOUNTAIN,33.649420,-115.856950
4,"MORTMAR, CALIFORNIA",33.521670,-115.935830
5,AREA 51,37.235000,-115.811110
6,CALVADA MEADOWS AIRPORT,36.271110,-115.995000
7,"KANE SPRING, CALIFORNIA",33.109440,-115.836670
8,HEXIE MOUNTAINS,33.886680,-115.955830
9,"ELMORE DESERT RANCH, CALIFORNIA",33.103610,-115.800560



`First 6 POIs:
- MOJAVE TRAILS NATIONAL MONUMENT: (0.60388, -2.02458)
- NORTH SHORE, CALIFORNIA: (0.58491, -2.02331)
- SQUEAKY SPRINGS, CALIFORNIA: (0.58046, -2.02368)
- BURIED MOUNTAIN: (0.58729, -2.02209)
- MORTMAR, CALIFORNIA: (0.58506, -2.02346)
- AREA 51: (0.64987, -2.02129)


In [6]:
import folium

m = folium.Map(location=[37.0, -115.7], zoom_start=6)

# Add a marker for each POI using degree columns
for idx, row in poi_nv.iterrows():
    lat = row["latitude_degree"]
    lon = row["longitude_degree"]
    name = row["name"]

    folium.Marker(
        location=[lat, lon],
        popup=name,
        tooltip=name,
        icon=folium.Icon(color="blue", icon="info-sign"),
    ).add_to(m)

# Display the map
m
